# Extract embedded layers for XStoryCloze

In [1]:
%pip install -q torch --index-url https://download.pytorch.org/whl/cu126
%pip install -q transformers accelerate tqdm huggingface_hub

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
assert torch.cuda.is_available(), "CUDA not available"
print(torch.__version__, "|", torch.cuda.get_device_name(0))

2.14.1+cu126 | NVIDIA GeForce RTX 4070 Ti


In [ ]:
import os, subprocess

REPO_URL = "https://github.com/CreamCheesePeanutButter/reproducing-cross-lingual-alignment.git"
parent = os.path.abspath("..")

if os.path.isfile(os.path.join(parent, "src_alignment", "embed_extractor.py")):
    REPO_DIR = parent   # running from <repo>/notebooks
else:
    REPO_DIR = os.path.abspath("reproducing-cross-lingual-alignment")
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
print("Repo ready at", REPO_DIR)

In [ ]:
from huggingface_hub import login, whoami
try:
    print("Logged in as", whoami()["name"])
except Exception:
    login()

In [4]:
MODEL = "meta-llama/Llama-3.2-1B"
NUM_SENTS = 504
OUT_ROOT = os.path.join(REPO_DIR, "embeddings", "meta-llama", "Llama-3.2-1B")
EXTRACTOR = os.path.join(REPO_DIR, "src_alignment", "embed_extractor.py")

In [5]:
import sys

def extract(kind):
    cmd = [
        sys.executable, EXTRACTOR,
        "--model_name", MODEL,
        "--data_path", os.path.join(REPO_DIR, "data", f"xstorycloze_{kind}"),
        "--file_ext", ".txt",
        "--num_sents", str(NUM_SENTS),
        "--gpus", "0",
        "--save_path", os.path.join(OUT_ROOT, f"xstorycloze_{kind}"),
    ]
    print(f"\n=== xstorycloze_{kind} ===")
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"Extraction failed for xstorycloze_{kind}")

extract("dali")


=== xstorycloze_dali ===
c:\issue21\.venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Twitc\.cache\huggingface\hub\models--meta-llama--Llama-3.2-1B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\issue21\.venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub

In [6]:
extract("mexa")


=== xstorycloze_mexa ===

Loading weights: 100%|██████████| 146/146 [00:01<00:00, 117.49it/s]
None
Extracting language: arabic
Extracting language: chinese
Extracting language: english
Extracting language: hindi
Extracting language: indonesian
Extracting language: russian
Extracting language: spanish

100%|██████████| 7/7 [01:08<00:00,  9.80s/it]


In [7]:
import glob, pickle

for kind, expected in [("dali", 14), ("mexa", 7)]:
    files = sorted(glob.glob(os.path.join(OUT_ROOT, f"xstorycloze_{kind}", "*.pkl")))
    print(f"xstorycloze_{kind}: {len(files)} files (expected {expected})")
    for f in files:
        print(f"  {os.path.basename(f):32s} {os.path.getsize(f)/1e6:6.1f} MB")

with open(files[0], "rb") as fh:
    d = pickle.load(fh)
print("\nlayers:", len(d), "| sentences in layer 0:", len(d[0]), "| vector dim:", d[0][0]["embd_lasttoken"].shape)

xstorycloze_dali: 14 files (expected 14)
  arabic_choice1.pkl                 70.6 MB
  arabic_choice2.pkl                 70.6 MB
  chinese_choice1.pkl                70.6 MB
  chinese_choice2.pkl                70.6 MB
  english_choice1.pkl                70.6 MB
  english_choice2.pkl                70.6 MB
  hindi_choice1.pkl                  70.6 MB
  hindi_choice2.pkl                  70.6 MB
  indonesian_choice1.pkl             70.6 MB
  indonesian_choice2.pkl             70.6 MB
  russian_choice1.pkl                70.6 MB
  russian_choice2.pkl                70.6 MB
  spanish_choice1.pkl                70.6 MB
  spanish_choice2.pkl                70.6 MB
xstorycloze_mexa: 7 files (expected 7)
  arabic.pkl                         70.6 MB
  chinese.pkl                        70.6 MB
  english.pkl                        70.6 MB
  hindi.pkl                          70.6 MB
  indonesian.pkl                     70.6 MB
  russian.pkl                        70.6 MB
  spanish.pkl       